# SG 最近 24h 行情与 TE 下单延迟比较

本报告只保留最近 24h 宏观比较口径：

1. `spread_pbs` 最近 24h 行情延迟，来自 SG 机器 pmdaemon 日志 grep。
2. `trade_engine` 下单链路延迟，来自 `bybit_intra_arb01/te_pubs/bybit/latency` 的 latency snapshot 采样。

机器：SG `ubuntu@47.131.162.78`。


## 行情延迟口径

行情部分参考当前 Bybit spread_pbs 进程：

- 进程：`/home/ubuntu/spread_pbs/bybit-both/spread_pbs --venue bybit-both --core 10`
- pmdaemon 名称：`spp_bb_bo`
- 日志：`/home/ubuntu/.pmdaemon/logs/spp_bb_bo-error.log`、`/home/ubuntu/.pmdaemon/logs/spp_bb_bo-out.log`
- 统计窗口：`2026-06-08T06:48:35Z` 到 `2026-06-09T06:48:35Z`
- 解析行数：53,400 条 `spread_pbs[...] latency_us` 日志窗口

主表采用 `spread_pbs[bybit-futures]` 与 `spread_pbs[bybit-margin]` 两个 e2e 标签，即 `accepted_us - event_ts_us`。当前 `spread_pbs` 日志只打印 `p90/p95/p99`，没有打印 `p50`，所以行情部分不从日志反推 p50。

## 行情核心结论

1. 期货和现货/杠杆现货常态窗口非常接近。
2. 期货 e2e 的 `median(window p90)` 为 2.677 ms，`median(window p99)` 为 3.095 ms。
3. 现货/杠杆现货 e2e 的 `median(window p90)` 为 2.613 ms，`median(window p99)` 为 3.003 ms。
4. 日志没有 p50；实时面板看到 p50 低于 3 ms 是合理的，因为 p50 必然不高于窗口 p90。
5. 对 Bybit intra 策略，信号层存在 BBO stale gate：超过 3000us 的 BBO 不会 `mark_dirty` 触发决策。也就是说，尾部过期行情会在信号触发层被过滤，不作为有效交易信号依据。
6. `spread_pbs` 日志按 venue 聚合 BTC/ETH/SOL，不区分单币种，也没有 `other` 样本。

## spread_pbs e2e 主表

| market | windows | samples | p50 | median(window p90) ms | p90(window p90) ms | median(window p99) ms | p90(window p99) ms |
|---|---:|---:|---|---:|---:|---:|---:|
| bybit-futures | 8,729 | 87,290,000 | N/A in logs | 2.677 | 2.918 | 3.095 | 4.545 |
| bybit-margin | 6,065 | 60,648,791 | N/A in logs | 2.613 | 2.831 | 3.003 | 4.427 |

表里的 `median(window p99)` 是所有日志窗口 p99 的中位数，不是把 24h 原始逐条样本重新合并后的全局 p99。日志里没有原始样本，所以只能按窗口分位数汇总。

## spread_pbs 全标签汇总

| market | metric | windows | samples | mean p90 ms | median p90 ms | p90(window p90) ms | mean p99 ms | median p99 ms | p90(window p99) ms |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|
| bybit-futures | e2e | 8,729 | 87,290,000 | 3.049 | 2.677 | 2.918 | 6.252 | 3.095 | 4.545 |
| bybit-futures | net | 8,729 | 87,290,000 | 3.046 | 2.673 | 2.914 | 6.248 | 3.091 | 4.541 |
| bybit-futures | ipc_e2e | 7,310 | 73,100,000 | 3.101 | 2.679 | 2.925 | 6.724 | 3.098 | 4.782 |
| bybit-futures | ipc_net | 7,310 | 73,100,000 | 3.097 | 2.675 | 2.921 | 6.721 | 3.094 | 4.778 |
| bybit-margin | e2e | 6,065 | 60,648,791 | 2.816 | 2.613 | 2.831 | 8.577 | 3.003 | 4.427 |
| bybit-margin | net | 6,065 | 60,648,791 | 2.812 | 2.609 | 2.828 | 8.574 | 2.999 | 4.424 |
| bybit-margin | ipc_e2e | 4,598 | 45,979,844 | 2.864 | 2.619 | 2.852 | 10.202 | 3.030 | 4.962 |
| bybit-margin | ipc_net | 4,598 | 45,979,844 | 2.860 | 2.615 | 2.848 | 10.198 | 3.026 | 4.959 |

`net` 为 `recv_us - event_ts_us`，`e2e` 为 `accepted_us - event_ts_us`。二者常态差值只有数微秒量级，说明 `spread_pbs` 内部解析/接受阶段不是主要延迟来源。

## 信号层过滤说明

在 `src/funding_rate/mkt_channel.rs` 中，Bybit intra 场景启用 `BYBIT_INTRA_BBO_STALE_GATE_US = 3000`。代码语义是：老于 3000us 的 BBO 仍会更新本地 quote cache，但不会 `mark_dirty_symbol`，因此不会触发新一轮信号决策。

所以报告中行情延迟的尾部窗口只用于评估数据链路稳定性，不应直接理解为这些 stale 行情都会进入交易触发路径。

## TE 下单延迟口径

TE 部分使用 `inspect_latency_snapshot` 订阅：

```bash
cd /home/ubuntu/bybit-intra-arb01
./inspect_latency_snapshot --service bybit_intra_arb01/te_pubs/bybit/latency --limit 10 --timeout-s 330 --poll-ms 20 --idle-log-s 0 --json
```

采样得到 10 个 30s snapshot，约 5 分钟窗口。`latency_stable_monitor` 最近 24h 日志确认持续收到 `bybit_intra_arb01_te` snapshot，但没有把 bucket 值落盘；因此 TE 部分用于和行情链路做同一量级的宏观比较。

本节不展示 `ipc_to_ws`，因为这段只有微秒级，对下单宏观延迟贡献很小。重点拆 `rtt` 的上行/下行：

- `uplink`：T2-T1，本地 WS 发送到交易所服务端时间戳。
- `downlink`：T4-T3，交易所服务端时间戳到本地收到响应。
- `rtt`：T4-T1，本地发送到本地收到响应的完整往返。
- `server`：T3-T2。Bybit 当前 snapshot 中为 0，说明响应只暴露一个服务端时间戳或服务端处理时间不可拆，本报告不单列。

注意：`uplink/downlink` 使用本地时钟和交易所服务端时间戳相减，会受两端时钟偏移影响；`rtt` 使用本地 monotonic elapsed，更适合作为完整下单响应耗时。

## TE 下单链路拆解

| action | windows | samples | uplink p50 ms | uplink p90 ms | uplink p99 ms | downlink p50 ms | downlink p90 ms | downlink p99 ms | rtt p50 ms | rtt p90 ms | rtt p99 ms |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| new | 10 | 943 | 1.417 | 1.857 | 2.414 | 1.530 | 1.940 | 2.317 | 2.924 | 3.233 | 4.138 |
| cancel | 10 | 914 | 1.249 | 1.706 | 2.265 | 1.528 | 1.913 | 2.398 | 2.740 | 3.131 | 3.982 |

这里的 p50/p90/p99 是 10 个 snapshot 窗口分位数的中位数，例如 `rtt p99 ms` 表示 `median(window p99)`。

观察：

- new 的 RTT p50 约 2.924ms，p90 约 3.233ms，p99 约 4.138ms。
- cancel 的 RTT p50 约 2.740ms，p90 约 3.131ms，p99 约 3.982ms。
- 上行和下行量级接近：new 上行 p50 1.417ms、下行 p50 1.530ms；cancel 上行 p50 1.249ms、下行 p50 1.528ms。
- RTT 大致等于上行 + 下行，符合 Bybit server 段为 0 的当前拆分。

## 可复跑代码：spread_pbs 日志


In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import re
import pandas as pd

LOG_PATHS = [
    Path('/home/ubuntu/.pmdaemon/logs/spp_bb_bo-error.log'),
    Path('/home/ubuntu/.pmdaemon/logs/spp_bb_bo-out.log'),
]
WINDOW_HOURS = 24

pattern = re.compile(
    r'^\[(?P<header>[^\]]+)\]\s+'
    r'spread_pbs\[(?P<label>[^\]]+)\]\s+'
    r'latency_us\s+n=(?P<n>\d+)\s+'
    r'p90=(?P<p90>-?\d+)\s+p95=(?P<p95>-?\d+)\s+p99=(?P<p99>-?\d+)'
)

def split_spread_label(label: str) -> tuple[str, str]:
    if label.endswith('-ipc-net'):
        return label[:-8], 'ipc_net'
    if label.endswith('-ipc'):
        return label[:-4], 'ipc_e2e'
    if label.endswith('-net'):
        return label[:-4], 'net'
    return label, 'e2e'

def parse_spread_pbs_latency_logs(log_paths=LOG_PATHS, hours=WINDOW_HOURS) -> pd.DataFrame:
    now = datetime.now(timezone.utc)
    cutoff = now - timedelta(hours=hours)
    rows = []
    for path in log_paths:
        if not path.exists():
            continue
        with path.open(errors='ignore') as f:
            for line in f:
                match = pattern.search(line)
                if not match:
                    continue
                ts_token = match.group('header').split()[0]
                ts = datetime.fromisoformat(ts_token.replace('Z', '+00:00'))
                if ts < cutoff:
                    continue
                market, metric = split_spread_label(match.group('label'))
                rows.append({
                    'ts': ts,
                    'market': market,
                    'metric': metric,
                    'n': int(match.group('n')),
                    'p90_us': int(match.group('p90')),
                    'p95_us': int(match.group('p95')),
                    'p99_us': int(match.group('p99')),
                })
    return pd.DataFrame(rows)

spread_df = parse_spread_pbs_latency_logs()
spread_df.head()


In [ ]:
def summarize_spread_windows(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (market, metric), sub in df.groupby(['market', 'metric']):
        rows.append({
            'market': market,
            'metric': metric,
            'windows': len(sub),
            'samples': int(sub['n'].sum()),
            'mean_p90_ms': sub['p90_us'].mean() / 1000,
            'median_p90_ms': sub['p90_us'].quantile(0.50) / 1000,
            'window_p90_p90_ms': sub['p90_us'].quantile(0.90) / 1000,
            'mean_p99_ms': sub['p99_us'].mean() / 1000,
            'median_p99_ms': sub['p99_us'].quantile(0.50) / 1000,
            'window_p90_p99_ms': sub['p99_us'].quantile(0.90) / 1000,
        })
    out = pd.DataFrame(rows).sort_values(['market', 'metric'])
    numeric = [c for c in out.columns if c.endswith('_ms')]
    out[numeric] = out[numeric].round(4)
    return out

spread_summary = summarize_spread_windows(spread_df)
spread_summary


## 可复跑代码：TE snapshot


In [ ]:
# 在 SG 机器上采样 TE latency snapshot：
# cd /home/ubuntu/bybit-intra-arb01
# ./inspect_latency_snapshot --service bybit_intra_arb01/te_pubs/bybit/latency --limit 10 --timeout-s 330 --poll-ms 20 --idle-log-s 0 --json > /tmp/te_latency_snapshot.jsonl

import json
from pathlib import Path
import pandas as pd

def parse_te_snapshot_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line.startswith('{'):
            continue
        snap = json.loads(line)
        for bucket in snap.get('buckets', []):
            rows.append({
                'seq': snap['seq'],
                'snapshot_time_us': snap['snapshot_time_us'],
                'metric': bucket['metric'],
                'action': bucket['action'],
                'n': bucket['n'],
                'p50_us': bucket['p50_us'],
                'p90_us': bucket['p90_us'],
                'p95_us': bucket['p95_us'],
                'p99_us': bucket['p99_us'],
            })
    return pd.DataFrame(rows)

# te_df = parse_te_snapshot_jsonl(Path('/tmp/te_latency_snapshot.jsonl'))


In [ ]:
def summarize_te_detail(te_df: pd.DataFrame) -> pd.DataFrame:
    keep = te_df[te_df['metric'].isin(['uplink', 'downlink', 'rtt'])].copy()
    rows = []
    for action, sub_action in keep.groupby('action'):
        row = {
            'action': action,
            'windows': int(sub_action['seq'].nunique()),
        }
        # samples are action-level; use rtt bucket as representative count.
        rtt = sub_action[sub_action['metric'] == 'rtt']
        row['samples'] = int(rtt['n'].sum())
        for metric in ['uplink', 'downlink', 'rtt']:
            sub = sub_action[sub_action['metric'] == metric]
            row[f'{metric}_p50_ms'] = sub['p50_us'].quantile(0.50) / 1000
            row[f'{metric}_p90_ms'] = sub['p90_us'].quantile(0.50) / 1000
            row[f'{metric}_p99_ms'] = sub['p99_us'].quantile(0.50) / 1000
        rows.append(row)
    out = pd.DataFrame(rows).sort_values('action')
    numeric = [c for c in out.columns if c.endswith('_ms')]
    out[numeric] = out[numeric].round(4)
    return out

# te_detail = summarize_te_detail(te_df)
# te_detail
